# Tokenization Deep Dive — Part 1: Fundamentals

---

## Session Outline

| # | Topic | Key Concepts |
|---|-------|--------------|
| 1 | **Why Tokenization Matters** | Text to Numbers pipeline |
| 2 | **Character-Level Tokenization** | Simplest approach, pros/cons |
| 3 | **Word-Level Tokenization** | Vocabulary explosion, OOV problem |
| 4 | **Subword Tokenization Concepts** | BPE, WordPiece, Unigram |
| 5 | **BPE from Scratch** | Full implementation |
| 6 | **WordPiece from Scratch** | Full implementation |

> **Prerequisites**: Python 3.8+  
> **Setup**: `pip install -r requirements.txt`

In [ ]:
# Setup
import re
import collections
from pprint import pprint

# Sample corpus we'll use throughout
corpus = [
    "Tokenization is the first step in any NLP pipeline.",
    "Tokens can be words, subwords, or characters.",
    "Subword tokenization balances vocabulary size and sequence length.",
    "BPE and WordPiece are the most popular subword methods.",
    "GPT uses BPE tokenization while BERT uses WordPiece.",
    "The tokenizer converts text into token IDs that the model understands.",
    "A good tokenizer should handle unseen words gracefully.",
    "Training a tokenizer means learning the best way to split text.",
    "Low frequency words get split into smaller subword units.",
    "High frequency words remain as single tokens."
]

print(f"Corpus has {len(corpus)} sentences")
print(f"Total characters: {sum(len(s) for s in corpus)}")

---
# Part 1 — Why Tokenization Matters

Every NLP model works with **numbers**, not raw text. Tokenization is the bridge:

```
"Hello world!" --> tokenizer --> [15496, 995, 0] --> model
```

### The fundamental question
> **How do we split text into atomic units (tokens) and map each to an integer?**

Different strategies yield vastly different vocabularies, sequence lengths, and model behaviour.

| Strategy | Vocab Size | Seq Length | OOV? | Used By |
|----------|-----------|------------|------|---------|
| Character | ~100 | Very long | No | Early RNNs |
| Word | 100k+ | Short | Yes | word2vec, GloVe |
| Subword (BPE) | 30k-50k | Moderate | Rare | GPT, LLaMA, Mistral |
| Subword (WordPiece) | 30k | Moderate | Rare | BERT, DistilBERT |
| SentencePiece (Unigram) | 32k | Moderate | Rare | T5, ALBERT |

---
# Part 2 — Character-Level Tokenization

The simplest strategy: treat every character as a token.

**Pros**: Tiny vocabulary (~100 chars), zero OOV words  
**Cons**: Very long sequences, each token carries little semantic meaning, harder for models to learn

In [ ]:
class CharTokenizer:
    """Character-level tokenizer."""
    
    def __init__(self):
        self.char_to_id = {}
        self.id_to_char = {}
    
    def train(self, texts):
        """Build vocabulary from a list of texts."""
        chars = set()
        for text in texts:
            chars.update(text)
        
        # Sort for deterministic ordering and add special tokens
        sorted_chars = sorted(chars)
        special_tokens = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
        
        for i, token in enumerate(special_tokens + sorted_chars):
            self.char_to_id[token] = i
            self.id_to_char[i] = token
    
    def encode(self, text):
        """Convert text to list of integer IDs."""
        unk_id = self.char_to_id["<UNK>"]
        return [self.char_to_id.get(ch, unk_id) for ch in text]
    
    def decode(self, ids):
        """Convert list of integer IDs back to text."""
        special = {"<PAD>", "<UNK>", "<BOS>", "<EOS>"}
        return "".join(self.id_to_char[i] for i in ids if self.id_to_char[i] not in special)
    
    @property
    def vocab_size(self):
        return len(self.char_to_id)


# Train and test
char_tok = CharTokenizer()
char_tok.train(corpus)

print(f"Vocabulary size: {char_tok.vocab_size}")
print(f"\nVocabulary:")
pprint(char_tok.char_to_id)

In [ ]:
# Encode / Decode demo
test_text = "Hello tokenizer!"
encoded = char_tok.encode(test_text)
decoded = char_tok.decode(encoded)

print(f"Original : {test_text}")
print(f"Encoded  : {encoded}")
print(f"Decoded  : {decoded}")
print(f"Seq len  : {len(encoded)}")
print(f"Roundtrip: {test_text == decoded}")

# Show what happens with unseen characters
unseen = "Hola mundo 42!"
enc_unseen = char_tok.encode(unseen)
print(f"\nUnseen text: {unseen}")
print(f"Encoded    : {enc_unseen}")
print(f"Decoded    : {char_tok.decode(enc_unseen)}")
print("Note: Characters not in training data map to <UNK> (id=1)")

### Key takeaway
Character tokenization **never** has OOV for characters it was trained on, but sequences become very long.
For the sentence above, we get **one token per character** — that's a lot of positions for a Transformer to attend over!

---
# Part 3 — Word-Level Tokenization

Split on whitespace (and maybe punctuation). Each unique word is a token.

**Pros**: Each token is semantically meaningful, short sequences  
**Cons**: Huge vocabulary, can't handle misspellings / new words (OOV)

In [ ]:
class WordTokenizer:
    """Simple whitespace + punctuation word-level tokenizer."""
    
    def __init__(self):
        self.word_to_id = {}
        self.id_to_word = {}
        # Regex: split on whitespace, keep punctuation as separate tokens
        self.pattern = re.compile(r"\w+|[^\w\s]")
    
    def train(self, texts):
        """Build vocabulary from texts."""
        special_tokens = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
        word_freq = collections.Counter()
        
        for text in texts:
            tokens = self.pattern.findall(text.lower())
            word_freq.update(tokens)
        
        # Build vocab ordered by frequency (most common first)
        all_tokens = special_tokens + [w for w, _ in word_freq.most_common()]
        for i, token in enumerate(all_tokens):
            self.word_to_id[token] = i
            self.id_to_word[i] = token
    
    def encode(self, text):
        unk_id = self.word_to_id["<UNK>"]
        tokens = self.pattern.findall(text.lower())
        return [self.word_to_id.get(t, unk_id) for t in tokens]
    
    def decode(self, ids):
        special = {"<PAD>", "<UNK>", "<BOS>", "<EOS>"}
        words = [self.id_to_word[i] for i in ids if self.id_to_word[i] not in special]
        # Simple rejoin (won't perfectly reconstruct punctuation spacing)
        return " ".join(words)
    
    @property
    def vocab_size(self):
        return len(self.word_to_id)


word_tok = WordTokenizer()
word_tok.train(corpus)

print(f"Vocabulary size: {word_tok.vocab_size}")
print(f"\nTop-20 tokens by frequency:")
for token, idx in list(word_tok.word_to_id.items())[:24]:
    print(f"  {idx:3d} -> '{token}'")

In [ ]:
# Encode / Decode demo
test_text = "Subword tokenization is the best approach."
encoded = word_tok.encode(test_text)
decoded = word_tok.decode(encoded)

print(f"Original : {test_text}")
print(f"Tokens   : {[word_tok.id_to_word[i] for i in encoded]}")
print(f"IDs      : {encoded}")
print(f"Decoded  : {decoded}")
print(f"Seq len  : {len(encoded)} (vs {len(char_tok.encode(test_text))} for char-level)")

# OOV problem
oov_text = "Transformerization is revolutionizing NLP."
oov_encoded = word_tok.encode(oov_text)
print(f"\nOOV test : {oov_text}")
print(f"Tokens   : {[word_tok.id_to_word[i] for i in oov_encoded]}")
print("Note: 'transformerization' and 'revolutionizing' map to <UNK>!")

### The OOV Problem

Word-level tokenization fails on:
- **New / rare words**: "transformerization" is <UNK>
- **Morphological variants**: "tokenize", "tokenizing", "tokenized" are all separate vocab entries
- **Misspellings**: "tokeniztaion" is <UNK>
- **Languages with rich morphology**: Turkish, Finnish, etc.

This motivates **subword tokenization**: a middle ground between characters and words.

---
# Part 4 — Subword Tokenization Concepts

### Core Idea
- Common words stay whole: `"the"` -> `["the"]`
- Rare words get split: `"unhappiness"` -> `["un", "happi", "ness"]`

### Three Main Algorithms

| Algorithm | Strategy | Used By |
|-----------|----------|---------|
| **BPE** (Byte Pair Encoding) | Bottom-up: start with characters, greedily merge most frequent pairs | GPT-2, GPT-3, GPT-4, LLaMA, Mistral |
| **WordPiece** | Similar to BPE, but merges by mutual information score instead of raw frequency | BERT, DistilBERT, Electra |
| **Unigram** | Top-down: start with large vocab, iteratively remove tokens that hurt likelihood least | T5, ALBERT, mBART |

Let's implement **BPE** and **WordPiece** from scratch!

---
# Part 5 — BPE (Byte Pair Encoding) from Scratch

### Algorithm
1. Start with a vocabulary of individual characters (+ special end-of-word token)
2. Count the frequency of every adjacent pair in the corpus
3. Merge the most frequent pair into a new token
4. Repeat steps 2-3 until desired vocabulary size is reached

### Encoding (at inference)
Apply learned merges in the same order they were learned.

In [ ]:
class BPETokenizer:
    """
    Byte Pair Encoding tokenizer built from scratch.
    Demonstrates the core BPE algorithm used by GPT-2/3/4.
    """
    
    def __init__(self, vocab_size=300):
        self.vocab_size = vocab_size
        self.merges = []          # List of (pair, merged_token) in order learned
        self.token_to_id = {}
        self.id_to_token = {}
    
    def _get_word_freqs(self, texts):
        """Tokenize at word level and count frequencies. Each word is split into characters + </w>."""
        word_freqs = collections.Counter()
        for text in texts:
            words = re.findall(r"\w+|[^\w\s]", text.lower())
            for word in words:
                # Represent each word as a tuple of characters + end-of-word marker
                word_freqs[tuple(list(word) + ["</w>"])] += 1
        return word_freqs
    
    def _get_pair_freqs(self, word_freqs):
        """Count frequency of adjacent symbol pairs across all words."""
        pair_freqs = collections.Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pair_freqs[(word[i], word[i + 1])] += freq
        return pair_freqs
    
    def _merge_pair(self, word_freqs, pair):
        """Merge all occurrences of `pair` in the word_freqs dictionary."""
        new_word_freqs = {}
        bigram = pair
        replacement = pair[0] + pair[1]
        
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == bigram[0] and word[i + 1] == bigram[1]:
                    new_word.append(replacement)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        
        return new_word_freqs
    
    def train(self, texts):
        """Learn BPE merges from training texts."""
        word_freqs = self._get_word_freqs(texts)
        
        # Initial vocabulary: all individual characters + </w> + special tokens
        vocab = set()
        for word in word_freqs:
            for symbol in word:
                vocab.add(symbol)
        
        print(f"Initial vocab size: {len(vocab)}")
        print(f"Initial vocab: {sorted(vocab)}\n")
        
        # Iteratively merge most frequent pairs
        num_merges = self.vocab_size - len(vocab) - 4  # Reserve space for special tokens
        
        for step in range(num_merges):
            pair_freqs = self._get_pair_freqs(word_freqs)
            if not pair_freqs:
                break
            
            best_pair = pair_freqs.most_common(1)[0]
            pair, freq = best_pair
            
            # Merge this pair
            merged = pair[0] + pair[1]
            self.merges.append(pair)
            vocab.add(merged)
            word_freqs = self._merge_pair(word_freqs, pair)
            
            if step < 15:  # Print first 15 merges
                print(f"Merge {step + 1:3d}: '{pair[0]}' + '{pair[1]}' -> '{merged}' (freq={freq})")
        
        # Build final vocab with special tokens
        special = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
        all_tokens = special + sorted(vocab)
        for i, token in enumerate(all_tokens):
            self.token_to_id[token] = i
            self.id_to_token[i] = token
        
        print(f"\n... ({len(self.merges)} merges total)")
        print(f"Final vocab size: {len(self.token_to_id)}")
    
    def _apply_merges(self, word):
        """Apply learned merges to a word (list of symbols)."""
        symbols = list(word) + ["</w>"]
        
        for pair in self.merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == pair[0] and symbols[i + 1] == pair[1]:
                    symbols = symbols[:i] + [pair[0] + pair[1]] + symbols[i + 2:]
                else:
                    i += 1
        return symbols
    
    def encode(self, text):
        """Encode text to token IDs."""
        words = re.findall(r"\w+|[^\w\s]", text.lower())
        ids = []
        tokens = []
        unk_id = self.token_to_id["<UNK>"]
        
        for word in words:
            subwords = self._apply_merges(word)
            for sw in subwords:
                tokens.append(sw)
                ids.append(self.token_to_id.get(sw, unk_id))
        
        return ids, tokens
    
    def decode(self, ids):
        """Decode token IDs back to text."""
        special = {"<PAD>", "<UNK>", "<BOS>", "<EOS>"}
        tokens = [self.id_to_token[i] for i in ids if self.id_to_token[i] not in special]
        text = "".join(tokens)
        text = text.replace("</w>", " ").strip()
        return text


# Train BPE
bpe_tok = BPETokenizer(vocab_size=200)
bpe_tok.train(corpus)

In [ ]:
# BPE Encoding Demo
test_sentences = [
    "Tokenization is important.",
    "The tokenizer splits text into tokens.",
    "Transformerization is a new word.",   # unseen word
]

for sent in test_sentences:
    ids, tokens = bpe_tok.encode(sent)
    decoded = bpe_tok.decode(ids)
    print(f"Input  : {sent}")
    print(f"Tokens : {tokens}")
    print(f"IDs    : {ids}")
    print(f"Decoded: {decoded}")
    print(f"Num tokens: {len(tokens)} (vs {len(sent)} chars, vs {len(sent.split())} words)")
    print()

### Observations
- Common words like "the", "is", "into" get merged into single tokens
- Rare words get split into subwords
- The unseen word "transformerization" is broken into known subword pieces
- **No OOV!** Every word can be decomposed into character-level pieces at worst

---
# Part 6 — WordPiece from Scratch

WordPiece (used by BERT) is similar to BPE but with a key difference:

**BPE** merges the most **frequent** pair.  
**WordPiece** merges the pair that maximizes **likelihood** of the training data (approximated by mutual information).

Score formula:
```
score(a, b) = freq(ab) / (freq(a) * freq(b))
```

Also, WordPiece uses `##` prefix to indicate continuation subwords.

In [ ]:
class WordPieceTokenizer:
    """
    WordPiece tokenizer built from scratch.
    Demonstrates the algorithm used by BERT.
    Uses ## prefix for continuation subwords.
    """
    
    def __init__(self, vocab_size=300):
        self.vocab_size = vocab_size
        self.vocab = set()
        self.token_to_id = {}
        self.id_to_token = {}
    
    def _get_word_freqs(self, texts):
        word_freqs = collections.Counter()
        for text in texts:
            words = re.findall(r"\w+|[^\w\s]", text.lower())
            for word in words:
                # First char stays as-is, rest get ## prefix
                symbols = tuple([word[0]] + [f"##{c}" for c in word[1:]])
                word_freqs[symbols] += 1
        return word_freqs
    
    def _compute_pair_scores(self, word_freqs):
        """Compute WordPiece scores: freq(ab) / (freq(a) * freq(b))."""
        # First, compute individual symbol frequencies
        symbol_freqs = collections.Counter()
        pair_freqs = collections.Counter()
        
        for word, freq in word_freqs.items():
            for i, symbol in enumerate(word):
                symbol_freqs[symbol] += freq
                if i < len(word) - 1:
                    pair_freqs[(word[i], word[i + 1])] += freq
        
        # Compute scores
        scores = {}
        for pair, freq in pair_freqs.items():
            scores[pair] = freq / (symbol_freqs[pair[0]] * symbol_freqs[pair[1]])
        
        return scores
    
    def _merge_pair(self, word_freqs, pair):
        new_word_freqs = {}
        # When merging, the second token's ## prefix is absorbed
        if pair[1].startswith("##"):
            replacement = pair[0] + pair[1][2:]  # Remove ## from second part
        else:
            replacement = pair[0] + pair[1]
        
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                    new_word.append(replacement)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        
        return new_word_freqs
    
    def train(self, texts):
        word_freqs = self._get_word_freqs(texts)
        
        # Initial vocab
        self.vocab = set()
        for word in word_freqs:
            for symbol in word:
                self.vocab.add(symbol)
        
        print(f"Initial vocab size: {len(self.vocab)}")
        
        num_merges = self.vocab_size - len(self.vocab) - 4
        
        for step in range(num_merges):
            scores = self._compute_pair_scores(word_freqs)
            if not scores:
                break
            
            best_pair = max(scores, key=scores.get)
            best_score = scores[best_pair]
            
            if best_pair[1].startswith("##"):
                merged = best_pair[0] + best_pair[1][2:]
            else:
                merged = best_pair[0] + best_pair[1]
            
            self.vocab.add(merged)
            word_freqs = self._merge_pair(word_freqs, best_pair)
            
            if step < 15:
                print(f"Merge {step + 1:3d}: '{best_pair[0]}' + '{best_pair[1]}' -> '{merged}' (score={best_score:.4f})")
        
        special = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
        all_tokens = special + sorted(self.vocab)
        for i, token in enumerate(all_tokens):
            self.token_to_id[token] = i
            self.id_to_token[i] = token
        
        print(f"\nFinal vocab size: {len(self.token_to_id)}")
    
    def encode(self, text):
        """Greedy longest-match-first encoding."""
        words = re.findall(r"\w+|[^\w\s]", text.lower())
        all_tokens = []
        all_ids = []
        unk_id = self.token_to_id["<UNK>"]
        
        for word in words:
            tokens = []
            start = 0
            while start < len(word):
                end = len(word)
                found = False
                while start < end:
                    substr = word[start:end]
                    if start > 0:
                        substr = "##" + substr
                    if substr in self.token_to_id:
                        tokens.append(substr)
                        found = True
                        break
                    end -= 1
                if not found:
                    tokens.append("<UNK>")
                    start += 1
                else:
                    start = end
            
            all_tokens.extend(tokens)
            all_ids.extend([self.token_to_id.get(t, unk_id) for t in tokens])
        
        return all_ids, all_tokens
    
    def decode(self, ids):
        special = {"<PAD>", "<UNK>", "<BOS>", "<EOS>"}
        tokens = [self.id_to_token[i] for i in ids if self.id_to_token[i] not in special]
        text = ""
        for token in tokens:
            if token.startswith("##"):
                text += token[2:]
            else:
                if text:
                    text += " "
                text += token
        return text


wp_tok = WordPieceTokenizer(vocab_size=200)
wp_tok.train(corpus)

In [ ]:
# WordPiece Encoding Demo
test_sentences = [
    "Tokenization is important.",
    "The tokenizer splits text into tokens.",
    "Transformerization is amazing.",
]

for sent in test_sentences:
    ids, tokens = wp_tok.encode(sent)
    decoded = wp_tok.decode(ids)
    print(f"Input  : {sent}")
    print(f"Tokens : {tokens}")
    print(f"IDs    : {ids}")
    print(f"Decoded: {decoded}")
    print()

### BPE vs WordPiece Summary

| Feature | BPE | WordPiece |
|---------|-----|-----------|
| Merge criterion | Frequency | Mutual information score |
| Continuation marker | None (uses end-of-word `</w>`) | `##` prefix |
| Used by | GPT family, LLaMA, Mistral | BERT family |
| Encoding | Apply merges in learned order | Greedy longest-match-first |

Both produce similar results in practice. The choice is mostly historical/ecosystem driven.

---
## Continue to Part 2

In **`02_transformers_and_training.ipynb`** we will:
1. Use HuggingFace's fast `tokenizers` library
2. Explore pre-trained tokenizers (GPT-2, BERT)
3. Train a custom BPE tokenizer on a real dataset
4. **Train a small GPT-style language model** end-to-end